In [ ]:
import os, gc, glob, json
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, recall_score, confusion_matrix

for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)  

ROOT         = r"C:\CODES__SSD512\Minor_Major_Pneumonia"
DATA_DIR     = os.path.join(ROOT, "Major", "ChestXRay2017", "chest_xray")
H5_DIR       = os.path.join(ROOT, "Major", "H5")
MINOR_H5_DIR = os.path.join(ROOT, "Minor", "H5")
PRED_DIR = os.path.join(H5_DIR, "ensemble_preds")
os.makedirs(PRED_DIR, exist_ok=True)

# LABELS: 1=PNEUMONIA, 0=NORMAL
LABEL_MAP    = {'NORMAL': 0, 'PNEUMONIA': 1}
FOLDER_ORDER = ['PNEUMONIA', 'NORMAL'] 

RANDOM_SEED  = 42
VAL_FRACTION = 0.10
TTA_VIEWS    = 5

In [ ]:
import tensorflow as tf

class ClassBalancedFocalLoss(tf.keras.losses.Loss):
    
    def __init__(self, alpha=0.25, gamma=2.0, name="focal_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.alpha = alpha
        self.gamma = gamma

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        loss = alpha_t * tf.math.pow((1 - p_t), self.gamma) * cross_entropy
        return tf.reduce_mean(loss)

    def get_config(self):
        config = super().get_config()
        config.update({
            "alpha": self.alpha,
            "gamma": self.gamma,
        })
        return config


class AdaMed(tf.keras.optimizers.Optimizer):
    def __init__(self, learning_rate=0.001, beta_1=0.9, beta_2=0.999, gamma_align=0.1, weight_decay=1e-4, epsilon=1e-7, name="AdaMed", **kwargs):
        super().__init__(name=name, **kwargs)
        self._set_hyper("learning_rate", kwargs.get("lr", learning_rate))
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.gamma_align = gamma_align
        self.weight_decay = weight_decay
        self.epsilon = epsilon
        
    def _create_slots(self, var_list):
        for var in var_list:
            self.add_slot(var, "m")
            self.add_slot(var, "v")
            
    def _resource_apply_dense(self, grad, var, apply_state=None):
        m = self.get_slot(var, "m")
        v = self.get_slot(var, "v")
        lr = self._get_hyper("learning_rate")
        
        dot_product = tf.reduce_sum(grad * m)
        norm_g = tf.norm(grad)
        norm_m = tf.norm(m)
        rho_t = tf.clip_by_value(dot_product / (norm_g * norm_m + self.epsilon), 0.0, 1.0)
        
        m_t = self.beta_1 * m + (1.0 - self.beta_1) * (1.0 + self.gamma_align * rho_t) * grad
        v_t = self.beta_2 * v + (1.0 - self.beta_2) * tf.square(grad)
        
        wd = self.weight_decay * (2.0 - rho_t)
        var_t = var - lr * (m_t / (tf.sqrt(v_t) + self.epsilon) + wd * var)
        
        from tensorflow.keras import backend as K
        m_update = K.update(m, m_t)
        v_update = K.update(v, v_t)
        var_update = K.update(var, var_t)
        return tf.group(m_update, v_update, var_update)
        
    def get_config(self):
        config = super().get_config()
        config.update({
            "learning_rate": self._serialize_hyperparameter("learning_rate"),
            "beta_1": self.beta_1, "beta_2": self.beta_2, "gamma_align": self.gamma_align,
            "weight_decay": self.weight_decay, "epsilon": self.epsilon,
        })
        return config

CUSTOM_OBJECTS = {'AdaMed': AdaMed, 'ClassBalancedFocalLoss': ClassBalancedFocalLoss}

In [ ]:
MODELS = [

    dict(name="E27_SSL_94",   path=os.path.join(H5_DIR, "custom_E27_CLAHE_Contrast_Xception_P4I3A3M7T4.keras"),clahe=True, scaling="unit",     output="p_normal"),
  
    dict(name="E27_v5",       path=os.path.join(H5_DIR, "custom_E27_CLAHE_Contrast_Xception_P4I3A3M7T4_L1PNEU.keras"),clahe=True, scaling="xception", output="p_pneu"),
    

    dict(name="E27_minor_Xception", path=os.path.join(MINOR_H5_DIR, "E27_CLAHE_Contrast_Xception_P4I3A3M7T4.keras"),
         clahe=True, scaling="auto", output="auto"),
    dict(name="E25_DenseNet121",    path=os.path.join(MINOR_H5_DIR, "E25_CLAHE_Contrast_DenseNet121_P4I3A3M5T4.keras"),
         clahe=True, scaling="auto", output="auto"),
    dict(name="E26_ResNet50",       path=os.path.join(MINOR_H5_DIR, "E26_CLAHE_Contrast_ResNet50_P4I3A3M6T4.keras"),
         clahe=True, scaling="auto", output="auto"),
    dict(name="E29_ResNet50",       path=os.path.join(MINOR_H5_DIR, "E29_CLAHE_Contrast_ResNet50_P4I3A3M6T5.keras"),
         clahe=True, scaling="auto", output="auto"),
    dict(name="E11_MobileNetV2",    path=os.path.join(MINOR_H5_DIR, "E11_CLAHE_Contrast_MobileNetV2_P4I3A2M3T4.h5"),
         clahe=True, scaling="auto", output="auto"),
]
found = []
for m in MODELS:
    if os.path.exists(m["path"]):
        found.append(m)
    else:
        print("skip (not found):", m["path"])
MODELS = found
print("Models to evaluate:", [m["name"] for m in MODELS])

skip (not found): C:\CODES__SSD512\Minor_Major_Pneumonia\Major\H5\custom_E27_CLAHE_Contrast_Xception_P4I3A3M7T4.keras
Models to evaluate: ['E27_v5', 'E27_minor_Xception', 'E25_DenseNet121', 'E26_ResNet50', 'E29_ResNet50', 'E11_MobileNetV2']


In [ ]:

def list_split(split):
    items = []
    for label in FOLDER_ORDER:
        path = os.path.join(DATA_DIR, split, label)
        for f in os.listdir(path):
            if f.lower().endswith(('.jpeg', '.jpg', '.png')):
                items.append((os.path.join(path, f), LABEL_MAP[label]))
    return items

def read_gray(p):
    a = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    return None if a is None else cv2.resize(a, (224, 224))

def load(items):
    xs, ys = [], []
    for p, y in items:
        a = read_gray(p)
        if a is not None:
            xs.append(a); ys.append(y)
    return np.stack(xs), np.array(ys)


train_x, train_y = load(list_split("train"))
idx = np.arange(len(train_y))
_, hold_idx = train_test_split(idx, test_size=VAL_FRACTION, stratify=train_y, random_state=RANDOM_SEED)
off_x, off_y = load(list_split("val"))
x_val = np.concatenate([off_x, train_x[hold_idx]]); y_val = np.concatenate([off_y, train_y[hold_idx]])
del train_x, off_x; gc.collect()
x_test, y_test = load(list_split("test"))
N_OFFICIAL = len(off_y)   
print(f"val {x_val.shape} (NORMAL={int((y_val==0).sum())}, PNEUMONIA={int((y_val==1).sum())}) | test {x_test.shape}")


val (579, 224, 224) (NORMAL=163, PNEUMONIA=416) | test (624, 224, 224)


In [ ]:
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
tta_gen = ImageDataGenerator(rotation_range=5, width_shift_range=0.05, height_shift_range=0.05,zoom_range=0.05, horizontal_flip=True)

def prep_images(gray, m):
    out = []
    for a in gray:
        if m["size"] != 224:
            a = cv2.resize(a, (m["size"], m["size"]))
        if m["clahe"]:
            a = cv2.convertScaleAbs(CLAHE.apply(a), alpha=1.2, beta=0)
        out.append(cv2.cvtColor(a, cv2.COLOR_GRAY2RGB))
    return np.stack(out).astype('uint8')

IMAGENET_MEAN_BGR = np.array([103.939, 116.779, 123.68], dtype='float32')
TORCH_MEAN = np.array([0.485, 0.456, 0.406], dtype='float32'); TORCH_STD = np.array([0.229, 0.224, 0.225], dtype='float32')

def scale(b, m):
    b = b.astype('float32'); mode = m["scaling"]
    if mode == "xception":            # also MobileNetV2 / EfficientNet "tf" mode
        return b / 127.5 - 1.0
    if mode == "caffe":               # ResNet50 / VGG preprocess_input
        return b[..., ::-1] - IMAGENET_MEAN_BGR
    if mode == "torch":               # DenseNet121 preprocess_input
        return (b / 255.0 - TORCH_MEAN) / TORCH_STD
    return b / 255.0                  # "unit"

def auto_configure(model, m):
    """Pick scaling and output direction using ONLY the 62 official val images (clean view)."""
    xo = prep_images(x_val[:N_OFFICIAL], m); yo = y_val[:N_OFFICIAL]
    modes = ["unit", "xception", "caffe", "torch"] if m["scaling"] == "auto" else [m["scaling"]]
    best = None
    for mode in modes:
        mm = dict(m, scaling=mode)
        p = np.concatenate([model.predict_on_batch(scale(xo[i:i+8], mm)).reshape(-1) for i in range(0, len(xo), 8)])
        auc = roc_auc_score(yo, p)
        if m["output"] == "p_normal": auc = 1 - auc
        score = max(auc, 1 - auc) if m["output"] == "auto" else auc
        if best is None or score > best[0]:
            best = (score, mode, auc)
    m["scaling"] = best[1]
    if m["output"] == "auto":
        m["output"] = "p_pneu" if best[2] >= 0.5 else "p_normal"
    print(f"  {m['name']}: scaling={m['scaling']}, output={m['output']}, official-val AUC={best[0]:.3f}")

def predict_tta(model, x, m, bs=8):
    total = np.zeros(len(x), dtype='float64')
    np.random.seed(RANDOM_SEED)
    for v in range(TTA_VIEWS):
        preds = []
        for i in range(0, len(x), bs):
            b = x[i:i + bs].astype('float32')
            if v > 0:
                b = np.stack([tta_gen.random_transform(im) for im in b])
            preds.append(model.predict_on_batch(scale(b, m)).reshape(-1))
        total += np.concatenate(preds)
    p = total / TTA_VIEWS
    return 1.0 - p if m["output"] == "p_normal" else p   # always return P(PNEUMONIA)

for m in MODELS:
    out_file = os.path.join(PRED_DIR, m["name"] + ".npz")
    if os.path.exists(out_file):
        print("cached:", m["name"]); continue
    tf.keras.backend.clear_session(); gc.collect()
    model = load_model(m["path"], custom_objects=CUSTOM_OBJECTS, compile=False)
    m["size"] = int(model.input_shape[1] or 224)
    auto_configure(model, m)
    json.dump({k: m[k] for k in ("scaling", "output", "size")}, open(out_file.replace(".npz", ".json"), "w"))
    pv = predict_tta(model, prep_images(x_val, m), m)
    pt = predict_tta(model, prep_images(x_test, m), m)
    np.savez(out_file, val=pv, test=pt)
    print(f"{m['name']:16s} val AUC={roc_auc_score(y_val, pv):.4f}  test AUC={roc_auc_score(y_test, pt):.4f}")
    del model; gc.collect()
tf.keras.backend.clear_session()

  E27_v5: scaling=xception, output=p_pneu, official-val AUC=0.993
E27_v5           val AUC=0.9980  test AUC=0.9739
  E27_minor_Xception: scaling=unit, output=p_normal, official-val AUC=1.000
E27_minor_Xception val AUC=0.9989  test AUC=0.9749
  E25_DenseNet121: scaling=xception, output=p_normal, official-val AUC=0.995
E25_DenseNet121  val AUC=0.9982  test AUC=0.9624
  E26_ResNet50: scaling=unit, output=p_normal, official-val AUC=0.992
E26_ResNet50     val AUC=0.9913  test AUC=0.9461
  E29_ResNet50: scaling=unit, output=p_normal, official-val AUC=0.990
E29_ResNet50     val AUC=0.9820  test AUC=0.9270
  E11_MobileNetV2: scaling=torch, output=p_normal, official-val AUC=0.997
E11_MobileNetV2  val AUC=0.9885  test AUC=0.9303


In [ ]:

THRESHOLD_ON = "official62"

def youden(y, p):
    if THRESHOLD_ON == "official62":
        y, p = y[:N_OFFICIAL], p[:N_OFFICIAL]
    f, t, th = roc_curve(y, p, pos_label=1)
    return float(np.clip(th[np.argmax(t - f)], 0.01, 0.99))

def row(name, pv, pt):
    t = youden(y_val, pv)
    b = (pt >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, b, labels=[0, 1]).ravel()
    acc05 = accuracy_score(y_test, (pt >= 0.5).astype(int))
    print(f"{name:32s} testAUC={roc_auc_score(y_test, pt):.4f}  acc@0.5={acc05:.4f}  "f"acc@val-t={accuracy_score(y_test, b):.4f} (t={t:.2f})  sens={tp/(tp+fn):.3f}  spec={tn/(tn+fp):.3f}")

P = {m["name"]: np.load(os.path.join(PRED_DIR, m["name"] + ".npz")) for m in MODELS}
print("=== Single models ===")
for n, d in P.items():
    row(n, d["val"], d["test"])

def ens(names, mode="mean"):
    vs = np.stack([P[n]["val"] for n in names]); ts = np.stack([P[n]["test"] for n in names])
    if mode == "rank":
        from scipy.stats import rankdata
        allp = np.concatenate([vs, ts], axis=1)
        allp = np.stack([rankdata(a) / allp.shape[1] for a in allp])
        vs, ts = allp[:, :len(y_val)], allp[:, len(y_val):]
    return vs.mean(0), ts.mean(0)

names = list(P)
if len(names) >= 2:
    print("\n=== Ensembles (threshold from validation) ===")
    row("ALL mean", *ens(names))
    row("ALL rank-mean", *ens(names, "rank"))
    from itertools import combinations
    for k in range(2, len(names)):
        for c in combinations(names, k):
            row(" + ".join(c)[:32], *ens(list(c)))


=== Single models ===
E27_v5                           testAUC=0.9739  acc@0.5=0.8862  acc@val-t=0.9103 (t=0.67)  sens=0.979  spec=0.795
E27_minor_Xception               testAUC=0.9749  acc@0.5=0.8734  acc@val-t=0.9263 (t=0.88)  sens=0.949  spec=0.889
E25_DenseNet121                  testAUC=0.9624  acc@0.5=0.8317  acc@val-t=0.8654 (t=0.70)  sens=0.985  spec=0.667
E26_ResNet50                     testAUC=0.9461  acc@0.5=0.8253  acc@val-t=0.8798 (t=0.80)  sens=0.926  spec=0.803
E29_ResNet50                     testAUC=0.9270  acc@0.5=0.7981  acc@val-t=0.8349 (t=0.53)  sens=0.962  spec=0.624
E11_MobileNetV2                  testAUC=0.9303  acc@0.5=0.7628  acc@val-t=0.7804 (t=0.64)  sens=0.979  spec=0.449

=== Ensembles (threshold from validation) ===
ALL mean                         testAUC=0.9711  acc@0.5=0.8397  acc@val-t=0.9183 (t=0.78)  sens=0.956  spec=0.855
ALL rank-mean                    testAUC=0.9713  acc@0.5=0.8638  acc@val-t=0.9071 (t=0.36)  sens=0.946  spec=0.842
E27_v5 + E2

### Diagnostic only: the best test accuracy any threshold could give


In [ ]:
def oracle(pt):
    ths = np.unique(pt)
    accs = [accuracy_score(y_test, (pt >= t).astype(int)) for t in ths]
    i = int(np.argmax(accs)); return accs[i], ths[i]

for n, d in P.items():
    a, t = oracle(d["test"]); print(f"{n:32s} oracle acc={a:.4f} at t={t:.3f}")
if len(names) >= 2:
    a, t = oracle(ens(names)[1]); print(f"{'ALL mean':32s} oracle acc={a:.4f} at t={t:.3f}")

E27_v5                           oracle acc=0.9311 at t=0.784
E27_minor_Xception               oracle acc=0.9311 at t=0.896
E25_DenseNet121                  oracle acc=0.9038 at t=0.921
E26_ResNet50                     oracle acc=0.8926 at t=0.900
E29_ResNet50                     oracle acc=0.8654 at t=0.617
E11_MobileNetV2                  oracle acc=0.8590 at t=0.900
ALL mean                         oracle acc=0.9215 at t=0.762


: 